In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BRONZE = "workspace.olist_bronze"
SILVER = "workspace.olist_silver"


def ler(tabela):
    """Lê uma tabela da bronze e remove as colunas de controle da carga."""
    return spark.table(f"{BRONZE}.{tabela}").drop("_ingestion_ts", "_source_file")


def gravar(df, tabela):
    """Grava na prata, com a data e hora do processamento, e mostra a contagem."""
    df = df.withColumn("_silver_ts", F.current_timestamp())
    df.write.mode("overwrite").saveAsTable(f"{SILVER}.{tabela}")
    linhas = spark.table(f"{SILVER}.{tabela}").count()
    print(f"✅ {tabela}: {linhas} linhas")

In [0]:
orders = ler("orders")

colunas_data = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for coluna in colunas_data:
    orders = orders.withColumn(coluna, F.to_timestamp(coluna))

gravar(orders, "orders")

✅ orders: 99441 linhas


In [0]:
order_items = (
    ler("order_items")
    .withColumn("order_item_id", F.col("order_item_id").cast("int"))
    .withColumn("shipping_limit_date", F.to_timestamp("shipping_limit_date"))
    .withColumn("price", F.col("price").cast("double"))
    .withColumn("freight_value", F.col("freight_value").cast("double"))
)

gravar(order_items, "order_items")

✅ order_items: 112650 linhas


In [0]:
order_payments = (
    ler("order_payments")
    .withColumn("payment_sequential", F.col("payment_sequential").cast("int"))
    .withColumn("payment_installments", F.col("payment_installments").cast("int"))
    .withColumn("payment_value", F.col("payment_value").cast("double"))
    .withColumn(
        "payment_installments",
        F.when(F.col("payment_installments") == 0, 1).otherwise(F.col("payment_installments")),
    )
)

gravar(order_payments, "order_payments")

✅ order_payments: 103886 linhas


In [0]:
order_reviews = (
    ler("order_reviews")
    .withColumn("review_score", F.col("review_score").cast("int"))
    .withColumn("review_creation_date", F.to_timestamp("review_creation_date"))
    .withColumn("review_answer_timestamp", F.to_timestamp("review_answer_timestamp"))
)

janela = Window.partitionBy("order_id").orderBy(
    F.col("review_answer_timestamp").desc(), F.col("review_id")
)

order_reviews = (
    order_reviews
    .withColumn("_ordem", F.row_number().over(janela))
    .filter(F.col("_ordem") == 1)
    .drop("_ordem")
)

gravar(order_reviews, "order_reviews")

✅ order_reviews: 98673 linhas


In [0]:
products = (
    ler("products")
    .withColumnRenamed("product_name_lenght", "product_name_length")
    .withColumnRenamed("product_description_lenght", "product_description_length")
    .withColumn("product_category_name", F.coalesce("product_category_name", F.lit("nao_informado")))
)

colunas_inteiras = ["product_name_length", "product_description_length", "product_photos_qty"]
colunas_decimais = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]
for coluna in colunas_inteiras:
    products = products.withColumn(coluna, F.col(coluna).cast("int"))
for coluna in colunas_decimais:
    products = products.withColumn(coluna, F.col(coluna).cast("double"))

traducao = ler("product_category_name_translation").select(
    "product_category_name", "product_category_name_english"
)

# Duas categorias não existem no arquivo de tradução original
faltantes = spark.createDataFrame(
    [
        ("pc_gamer", "pc_gamer"),
        ("portateis_cozinha_e_preparadores_de_alimentos", "portable_kitchen_food_preparers"),
    ],
    ["product_category_name", "product_category_name_english"],
)
traducao = traducao.unionByName(faltantes)

products = (
    products.join(traducao, "product_category_name", "left")
    .withColumn(
        "product_category_name_english",
        F.coalesce("product_category_name_english", F.lit("not_defined")),
    )
)

gravar(products, "products")

✅ products: 32951 linhas


In [0]:
customers = ler("customers").withColumn(
    "customer_zip_code_prefix", F.col("customer_zip_code_prefix").cast("int")
)
gravar(customers, "customers")

sellers = ler("sellers").withColumn(
    "seller_zip_code_prefix", F.col("seller_zip_code_prefix").cast("int")
)
gravar(sellers, "sellers")

✅ customers: 99441 linhas
✅ sellers: 3095 linhas


In [0]:
geolocation = (
    ler("geolocation")
    .withColumn("geolocation_zip_code_prefix", F.col("geolocation_zip_code_prefix").cast("int"))
    .withColumn("geolocation_lat", F.col("geolocation_lat").cast("double"))
    .withColumn("geolocation_lng", F.col("geolocation_lng").cast("double"))
    .filter(
        F.col("geolocation_lat").between(-34, 6)
        & F.col("geolocation_lng").between(-74, -34)
    )
    .groupBy("geolocation_zip_code_prefix")
    .agg(
        F.median("geolocation_lat").alias("geolocation_lat"),
        F.median("geolocation_lng").alias("geolocation_lng"),
        F.min("geolocation_city").alias("geolocation_city"),
        F.min("geolocation_state").alias("geolocation_state"),
    )
)

gravar(geolocation, "geolocation")

✅ geolocation: 19010 linhas
